<a href="https://colab.research.google.com/github/jproney/ProteinEBM/blob/main/proteinebm_diffusion.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#@title Install ProteinEBM

import sys
import os

# Clone repository
if not os.path.exists('ProteinEBM'):
    os.system("pip install -q ml_collections biopython mdtraj py3Dmol")
    os.system("pip install -q git+https://github.com/sokrypton/py2Dmol.git")
    print("✓ Installed missing packages")
    os.system("git clone https://github.com/jproney/ProteinEBM.git")
    print("✓ Cloned ProteinEBM repository")
else:
    print("✓ ProteinEBM already cloned")

# Add to Python path (no pip install needed!)
sys.path.insert(0, '/content/ProteinEBM')

# Direct download from Hugging Face
expert_model_url = "https://huggingface.co/jproney/ProteinEBM/resolve/main/model_6_expert_frozen_1m_md.pt"
expert_model_path = "weights/model_6_expert_frozen_1m_md.pt"


if not os.path.exists(expert_model_path):
    os.makedirs('weights', exist_ok=True)
    print("Downloading model from Hugging Face...")
    !wget -q {expert_model_url} -O {expert_model_path}
    print(f"✓ Downloaded to {expert_model_path}")
else:
    print(f"✓ Model already exists")

# Direct download from Hugging Face
base_model_url = "https://huggingface.co/jproney/ProteinEBM/resolve/main/model_1_frozen_1m_md.pt"
base_model_path = "weights/model_1_frozen_1m_md.pt"


if not os.path.exists(base_model_path):
    os.makedirs('weights', exist_ok=True)
    print("Downloading model from Hugging Face...")
    !wget -q {base_model_url} -O {base_model_path}
    print(f"✓ Downloaded to {base_model_path}")
else:
    print(f"✓ Model already exists")

# Verify it works
try:
  if "model" not in dir():
      import torch
      import yaml
      import numpy as np
      from ml_collections import ConfigDict
      import py2Dmol
      from protein_ebm.model.r3_diffuser import R3Diffuser
      from protein_ebm.model.ebm import ProteinEBM
      from protein_ebm.model.boltz_utils import center_random_augmentation
      from protein_ebm.data.protein_utils import residues_to_features

      # Load config
      with open("ProteinEBM/protein_ebm/config/expert_model_config.yaml", 'r') as f:
          config = yaml.safe_load(f)
      config = ConfigDict(config)

      # Create model
      diffuser = R3Diffuser(config.diffuser)
      x_model = ProteinEBM(config.model, diffuser).cuda()

      # Load weights
      ckpt = torch.load(expert_model_path, weights_only=False, map_location='cuda')
      x_model.load_state_dict({k[len("model."):]: v for k, v in ckpt['state_dict'].items() if k.startswith('model')})
      x_model.eval()

      # Load config
      with open("ProteinEBM/protein_ebm/config/base_model_config.yaml", 'r') as f:
          config = yaml.safe_load(f)
      config = ConfigDict(config)


      base_model = ProteinEBM(config.model, diffuser).cuda()

      # Load weights
      ckpt = torch.load(base_model_path, weights_only=False, map_location='cuda')
      base_model.load_state_dict({k[len("model."):]: v for k, v in ckpt['state_dict'].items() if k.startswith('model')})
      base_model.eval()


  print("✓ ProteinEBM models loaded")
except ImportError as e:
    print(f"✗ Error: {e}")


def kabsch_rmsd(coords1, coords2):
    """Compute RMSD after optimal Kabsch alignment."""
    c1 = coords1.squeeze().cpu().numpy()
    c2 = coords2.squeeze().cpu().numpy()

    # Center
    c1_centered = c1 - c1.mean(axis=0)
    c2_centered = c2 - c2.mean(axis=0)

    # Kabsch algorithm
    H = c1_centered.T @ c2_centered
    U, S, Vt = np.linalg.svd(H)
    R = Vt.T @ U.T

    # Ensure proper rotation
    if np.linalg.det(R) < 0:
        Vt[-1, :] *= -1
        R = Vt.T @ U.T

    # Compute RMSD
    c1_aligned = c1_centered @ R
    rmsd = np.sqrt(((c1_aligned - c2_centered)**2).mean())

    return rmsd

# Running A Trajectory

- This notebook runs trajectories via Langevin Annealing, where we take multiple Langevin equilibration steps at each noise level.
- The number of steps at each noise level is controled by `LANGEVIN_PER_LEVEL`.
- The number of noise levels sampled is controlled by `REVERSE_STEPS`.
- To run a standard reverse diffusion trajectory, set `T_MAX=1`, `T_MIN` to a low value (i.e., .01), `REVERSE_STEPS=100-200`, `LANGEVIN_PER_LEVEL=0`
- To run a direct folding simulation, set `LANGEVIN_PER_LEVEL` to a large number (i.e., 10000), set `T_MAX=T_MIN={.05, 0.1}`, and make sure to check `start_unfolded`.

Note that, when transitioning between noise levels, the model energy may actually increase. This is not an issue, since relative energies are only meaningful within the same noise level. When running a direct folding simulation, energies should generally decrease over the course of the trajectory.

Note that, since ProteinEBM does not use MSAs, its structure predictions from reverse diffusion and folding simulations are not always correct! However, if you draw lots of samples and rank them by energy you should get the native structure more often. If you want to see your protein fold, you should probably run many trajectories in parallel. For code to do this and other applications of ProteinEBM see: https://github.com/jproney/ProteinEBM

In [ ]:
SEQUENCE = "MQIFVKTLTGKTITLEVEPSDTIENVKAKIQDKEGIPPDQQRLIFAGKQLEDGRTLSDYNIQKESTLHLVLRLRGG"  # @param {type:"string"}
PDB_ID = "1MI0" # @param {type:"string"}
REVERSE_STEPS = 1 # @param ["1","50","100","200","400","800"] {"type":"raw"}
LANGEVIN_PER_LEVEL = 10000 # @param {"type":"raw"}
start_unfolded = True # @param {"type":"boolean"}
use_aux_score = True # @param {"type":"boolean"}
T_MAX = .05 # @param {"type":"raw"}
T_MIN = .05 # @param {"type":"raw"}

# Convert sequence to features
from protein_ebm.data.protein_utils import restype_order, restype_num, generate_random_backbone_coords, residues_to_features
import Bio.PDB
import requests
import numpy as np
from Bio.PDB.Polypeptide import is_aa

def seq_to_features(seq):
    nres = len(seq)
    aatype = torch.tensor([restype_order.get(aa, restype_num) for aa in seq.upper()], dtype=torch.long)
    atom_mask = torch.zeros(nres, 37, dtype=torch.float32)
    atom_mask[:, 1] = 1.0 # Only CA atoms are considered for atom_mask_gen here
    residue_idx = torch.arange(nres, dtype=torch.long)
    residue_mask = torch.ones(nres, dtype=torch.float32)
    chain_encoding = torch.zeros_like(aatype)
    return aatype, atom_mask, residue_idx, residue_mask, chain_encoding

def download_pdb(pdb_id, pdb_dir="pdb_files"):
    os.makedirs(pdb_dir, exist_ok=True)
    pdb_file = os.path.join(pdb_dir, f'{pdb_id.lower()}.pdb')
    if not os.path.exists(pdb_file):
        print(f"Downloading PDB {pdb_id}...")
        url = f"https://files.rcsb.org/download/{pdb_id.upper()}.pdb"
        response = requests.get(url)
        response.raise_for_status()
        with open(pdb_file, 'w') as f:
            f.write(response.text)
        print(f"Downloaded {pdb_id} to {pdb_file}")
    else:
        print(f"PDB {pdb_id} already exists at {pdb_file}")
    return pdb_file

def parse_pdb_and_get_features(pdb_file):
    parser = Bio.PDB.PDBParser(QUIET=True)
    structure = parser.get_structure('PDB_ID', pdb_file)
    model = structure[0] # Assuming first model
    chain = [c for c in structure.get_chains()][0]

    # Convert Bio.PDB residues to protein_ebm features
    atom_positions, atom_mask, aatype, residue_idx = residues_to_features([r for r in chain.get_residues() if is_aa(r)])
    ca_coords = atom_positions[...,1,:]

    aatype_pdb = torch.tensor(aatype, dtype=torch.long)
    atom_mask_pdb = torch.tensor(atom_mask, dtype=torch.float32)
    residue_idx_pdb = torch.tensor(residue_idx, dtype=torch.long)
    residue_mask_pdb = torch.ones(atom_positions.shape[0], dtype=torch.float32)
    chain_encoding_pdb = torch.zeros_like(aatype_pdb)

    return aatype_pdb, atom_mask_pdb, residue_idx_pdb, residue_mask_pdb, chain_encoding_pdb, ca_coords

# Initialize variables that will hold the generated features and coordinates
aatype_gen, atom_mask_gen, residue_idx_gen, residue_mask_gen = None, None, None, None
chain_encoding_gen = None
nres = 0
pos_t, prev0 = None, None
initial_coords_no_noise = None # To store the un-noised base structure

if PDB_ID:
    print(f'PDB ID {PDB_ID} provided. Loading from PDB.')
    try:
        pdb_file_path = download_pdb(PDB_ID)
        aatype_gen, atom_mask_gen, residue_idx_gen, residue_mask_gen, chain_encoding_gen, ca_coords_initial = parse_pdb_and_get_features(pdb_file_path)
        nres = len(aatype_gen)

        # Create a reverse mapping from index to amino acid symbol for sequence generation
        rev_restype_order = {v: k for k, v in restype_order.items()}
        pdb_sequence = "".join([rev_restype_order.get(idx.item(), 'X') for idx in aatype_gen])

        if start_unfolded:
            # If start_unfolded is True, generate unfolded coordinates based on PDB sequence
            print(f"Using unfolded initialization from PDB sequence (derived from {PDB_ID}).")
            unfolded_coords, log_prob = generate_random_backbone_coords(pdb_sequence, uniform_sampling=True)
            unfolded_ca = unfolded_coords[:, 1, :].clone().unsqueeze(0)  # CA atoms only, [1, N, 3]
            initial_coords_no_noise = center_random_augmentation(
                unfolded_ca,
                torch.ones([1, nres], device=unfolded_ca.device),
                rotate=False
            ).cuda().view([1, -1, 3])
        else:
            # Otherwise, use coordinates directly from PDB
            ca_coords_initial_tensor = ca_coords_initial.unsqueeze(0).cuda()
            initial_coords_no_noise = center_random_augmentation(
                ca_coords_initial_tensor,
                torch.ones([1, nres], device=ca_coords_initial_tensor.device),
                rotate=False
            ).view([1, -1, 3])
            print(f"Loaded PDB {PDB_ID} with {nres} residues.")

    except Exception as e:
        print(f"Error loading PDB {PDB_ID}: {e}")
        print("Falling back to sequence-based generation.")
        PDB_ID = "" # Reset PDB_ID to trigger sequence logic below


if not PDB_ID: # This block will execute if PDB_ID was empty initially or if PDB loading failed
    print(f"Using sequence-based generation for protein: {SEQUENCE}.")
    aatype_gen, atom_mask_gen, residue_idx_gen, residue_mask_gen, chain_encoding_gen = seq_to_features(SEQUENCE)
    nres = len(aatype_gen)

    if start_unfolded:
        unfolded_coords, log_prob = generate_random_backbone_coords(SEQUENCE, uniform_sampling=True)
        unfolded_ca = unfolded_coords[:, 1, :].clone().unsqueeze(0)  # CA atoms only, [1, N, 3]
        # Center the structure and store as initial_coords_no_noise
        initial_coords_no_noise = center_random_augmentation(
            unfolded_ca,
            torch.ones([1, nres], device=unfolded_ca.device),
            rotate=False
        ).cuda().view([1, -1, 3])

# --- Conditional logic for pos_t initialization based on T_MAX ---

if T_MAX == 1.0:
    pos_t = torch.randn(1, nres, 3).cuda() * diffuser.config.coordinate_scaling
    # Ensure pos_t is centered even for random noise start
    pos_t = center_random_augmentation(
        pos_t.reshape([1, -1, 3]),
        torch.ones([1, nres], device=pos_t.device),
        rotate=False
    ).view([1, -1, 3])

elif T_MAX < 1.0:
    if initial_coords_no_noise is None:
        raise ValueError("A base structure (from PDB or unfolded sequence) must be provided when T_START < 1.0.")

    # Apply forward marginal diffusion to the base structure to reach T_MAX
    # Convert initial_coords_no_noise to numpy array on CPU before passing to diffuser
    pos_t_numpy = diffuser.forward_marginal(initial_coords_no_noise.cpu().numpy(), T_MAX)
    pos_t = torch.from_numpy(pos_t_numpy[0]).float().cuda() # Access the first element of the tuple and cast to float32

    # Ensure pos_t is centered after applying noise
    pos_t = center_random_augmentation(
        pos_t.reshape([1, -1, 3]),
        torch.ones([1, nres], device=pos_t.device),
        rotate=False
    ).view([1, -1, 3])
else:
    # Fallback for unexpected T_START > 1.0, treat as random start
    print("Warning: T_MAX > 1.0 is unusual. Starting with random noise.")
    pos_t = torch.randn(1, nres, 3).cuda() * diffuser.config.coordinate_scaling
    pos_t = center_random_augmentation(
        pos_t.reshape([1, -1, 3]),
        torch.ones([1, nres], device=pos_t.device),
        rotate=False
    ).view([1, -1, 3])

# Initialize prev0 based on instructions
if initial_coords_no_noise is not None and T_MAX < 1.0:
    prev0 = pos_t.clone() # Instruction: Initialize prev0 to pos_t.clone() if T_START < 1.0 and initial_coords_no_noise was used
else:
    prev0 = torch.zeros_like(pos_t) # Otherwise, keep prev0 = torch.zeros_like(pos_t)


# Viewer (moved outside if/else to ensure it's always initialized)
viewer = py2Dmol.view(scatter=True,
                      detect_cyclic=False)
viewer.new_obj(scatter_config={"xlabel": "Step", "ylabel": "Energy"})
viewer.show()

# Run
reverse_times = np.linspace(T_MIN, T_MAX, REVERSE_STEPS)[::-1] # Updated to use T_START and T_END
dt_rev = (T_MAX - T_MIN) / REVERSE_STEPS # Updated to use T_START and T_END
dt_lang = 0.001
frame_count = 0

# Ensure all features are on CUDA and properly shaped for input_feats
if aatype_gen is not None:
    aatype_gen_gpu = aatype_gen.unsqueeze(0).cuda()
    atom_mask_gen_gpu = atom_mask_gen.unsqueeze(0).cuda()
    residue_idx_gen_gpu = residue_idx_gen.unsqueeze(0).cuda()
    residue_mask_gen_gpu = residue_mask_gen.unsqueeze(0).cuda()
    chain_encoding_gen_gpu = chain_encoding_gen.unsqueeze(0).cuda()
else:
    # This block should ideally not be reached if PDB/sequence parsing is successful
    raise RuntimeError("Feature generation failed, aatype_gen is None.")

with torch.no_grad():
    for time_idx, t in enumerate(reverse_times):

        if t < .1:
          model = x_model
        else:
          model = base_model

        # LANGEVIN ANNEALING at this time level
        for lang_step in range(LANGEVIN_PER_LEVEL):
            input_feats = {
                'r_noisy': pos_t,
                'aatype': aatype_gen_gpu,
                'mask': residue_mask_gen_gpu,
                'residue_idx': residue_idx_gen_gpu,
                't': torch.tensor([t], dtype=torch.float).cuda(),
                'chain_encoding': chain_encoding_gen_gpu,
            }

            if use_aux_score:
              out = model.compute_energy(input_feats)
              score = out['r_update_aux']
              energy = out['energy']
              pred_coords = out['pred_coords_aux']
            else:
              out = model.compute_score(input_feats)
              score = out['trans_score']
              energy = out['energy']
              pred_coords = out['pred_coords']

            # Backward
            drift = diffuser.drift_coef(pos_t, t)
            diffusion = diffuser.diffusion_coef(t)
            pos_t = pos_t - (drift - diffusion**2 * score  / diffuser.config.coordinate_scaling
                            ) * dt_lang + diffusion * np.sqrt(dt_lang) * torch.randn_like(pos_t) / diffuser.config.coordinate_scaling

            # Center
            pos_t, prev0 = center_random_augmentation(
                pos_t.reshape([1, -1, 3]), torch.ones([1, nres], device=pos_t.device),
                rotate=False, second_coords=pred_coords.reshape([1, -1, 3]),
                return_second_coords=True
            )
            pos_t, prev0 = pos_t.view([1, -1, 3]), prev0.view([1, -1, 3])

            # Forward
            pos_t = pos_t + diffuser.drift_coef(pos_t, t - dt_lang) * dt_lang + \
                    diffuser.diffusion_coef(t - dt_lang) * np.sqrt(dt_lang) * \
                    torch.randn_like(pos_t) / diffuser.config.coordinate_scaling

            # ADD FRAME for every Langevin step
            viewer.add(prev0.cpu().squeeze(0).numpy(), scatter=[frame_count, energy.cpu().item()])
            frame_count += 1

        # REVERSE DIFFUSION STEP
        input_feats = {
            'r_noisy': pos_t,
            'aatype': aatype_gen_gpu,
            'mask': residue_mask_gen_gpu,
            'residue_idx': residue_idx_gen_gpu,
            't': torch.tensor([t], dtype=torch.float).cuda(),
            'chain_encoding': chain_encoding_gen_gpu,
            'atom_mask': atom_mask_gen_gpu
        }

        if use_aux_score:
          out = model.compute_energy(input_feats)
          score = out['r_update_aux']
          energy = out['energy']
        else:
          out = model.compute_score(input_feats)
          score = out['trans_score']
          energy = out['energy']

        pos_t = pos_t - (diffuser.drift_coef(pos_t, t) -
                        diffuser.diffusion_coef(t)**2 * score / diffuser.config.coordinate_scaling
                       ) * dt_rev + diffuser.diffusion_coef(t) * np.sqrt(dt_rev) * torch.randn_like(pos_t) / diffuser.config.coordinate_scaling

        pos_t, prev0 = center_random_augmentation(
            pos_t.reshape([1, -1, 3]), torch.ones([1, nres], device=pos_t.device),
            rotate=False, second_coords=pred_coords.reshape([1, -1, 3]),
            return_second_coords=True
        )
        pos_t, prev0 = pos_t.view([1, -1, 3]), prev0.view([1, -1, 3])

        # ADD FRAME for every reverse step
        viewer.add(prev0.cpu().squeeze(0).numpy(), scatter=[frame_count, energy.cpu().item()], align=False)
        frame_count += 1